In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DATASET_PATH = "/content/drive/MyDrive/lfm_poke_train_50k.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/lfm25_encoder_checkpoint"

assert os.path.exists(DATASET_PATH), "Dataset file not found in Google Drive!"

Mounted at /content/drive


In [ ]:
# Cell 3: Data Tokenization & Custom Classification Model
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from transformers.modeling_outputs import SequenceClassifierOutput

ACTION_LABELS = [
    "MOVE_1", "MOVE_2", "MOVE_3", "MOVE_4",
    "SWITCH_1", "SWITCH_2", "SWITCH_3", "SWITCH_4", "SWITCH_5"
]
label2id = {label: idx for idx, label in enumerate(ACTION_LABELS)}
id2label = {idx: label for idx, label in enumerate(ACTION_LABELS)}

# 1. Load Dataset
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.train_test_split(test_size=0.10, seed=42)

MODEL_ID = "LiquidAI/LFM2.5-Encoder-350M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)

def preprocess(batch):
    return tokenizer(batch["text"], max_length=256, truncation=True)

tokenized = dataset.map(preprocess, batched=True)
tokenized = tokenized.rename_column("label_id", "labels")

# 2. Define Custom LFM2.5 Sequence Classification Wrapper
class LFM25ForSequenceClassification(nn.Module):
    def __init__(self, model_id, num_labels, config):
        super().__init__()
        self.num_labels = num_labels
        self.config = config

        # Load the base bidirectional encoder body
        self.encoder = AutoModel.from_pretrained(model_id, config=config, trust_remote_code=True)
        hidden_size = config.hidden_size  # 1024 for LFM2.5-Encoder-350M

        # Classification head: Dropout -> Dense -> Tanh -> Classifier
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        # Forward pass through LFM2.5 bidirectional backbone
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)

        # outputs[0] has shape [batch_size, seq_len, hidden_size]
        last_hidden_state = outputs[0]

        # Mean Pooling over active non-padded tokens
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states if hasattr(outputs, "hidden_states") else None,
            attentions=outputs.attentions if hasattr(outputs, "attentions") else None,
        )

# 3. Instantiate Configuration and Custom Model
config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
config.num_labels = len(ACTION_LABELS)
config.label2id = label2id
config.id2label = id2label

model = LFM25ForSequenceClassification(
    model_id=MODEL_ID,
    num_labels=len(ACTION_LABELS),
    config=config
)

# 4. Metrics Setup
acc_metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=preds, references=labels)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="/content/runs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8),
    compute_metrics=compute_metrics
)

# 6. Execute Training & Save
trainer.train()

# Save final checkpoint (Model state + Config + Tokenizer)
torch.save(model.state_dict(), f"{OUTPUT_DIR}/pytorch_model.bin")
config.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Checkpoint permanently saved to: {OUTPUT_DIR}")

modeling_lfm2_bidirectional.py:   0%|          | 0.00/7.09k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/LFM2.5-Encoder-350M:
- modeling_lfm2_bidirectional.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights: 0it [00:00, ?it/s]

[transformers] Lfm2BidirectionalModel LOAD REPORT from: LiquidAI/LFM2.5-Encoder-350M
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
lfm2.layers.{2, 5, 8, 10, 12, 14}.self_attn.q_layernorm.weight     | UNEXPECTED | 
lfm2.layers.{2, 5, 8, 10, 12, 14}.self_attn.k_layernorm.weight     | UNEXPECTED | 
lfm2.layers.{0...15}.operator_norm.weight                          | UNEXPECTED | 
lfm2.layers.{0...15}.feed_forward.w3.weight                        | UNEXPECTED | 
lfm2.layers.{0, 1, 3, 4, 6, 7, 9, 11, 13, 15}.conv.conv.weight     | UNEXPECTED | 
lfm2.layers.{0...15}.feed_forward.w2.weight                        | UNEXPECTED | 
lfm2.layers.{2, 5, 8, 10, 12, 14}.self_attn.v_proj.weight          | UNEXPECTED | 
lfm2.layers.{0...15}.ffn_norm.weight                               | UNEXPECTED | 
lfm2.layers.{2, 5, 8, 10, 12, 14}.self_attn.k_proj.weight          | UNEXPECTED | 
lf

KeyError: 'eval'

In [ ]:
# Cell 3: Data Tokenization, Weight-Aligned Architecture, & Training
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from transformers.modeling_outputs import SequenceClassifierOutput

ACTION_LABELS = [
    "MOVE_1", "MOVE_2", "MOVE_3", "MOVE_4",
    "SWITCH_1", "SWITCH_2", "SWITCH_3", "SWITCH_4", "SWITCH_5"
]
label2id = {label: idx for idx, label in enumerate(ACTION_LABELS)}
id2label = {idx: label for idx, label in enumerate(ACTION_LABELS)}

# -------------------------------------------------------------
# 1. Dataset Loading & Fix KeyError ('train' / 'test')
# -------------------------------------------------------------
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# train_test_split produces splits named "train" and "test"
split_datasets = dataset.train_test_split(test_size=0.10, seed=42)

MODEL_ID = "LiquidAI/LFM2.5-Encoder-350M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)

def preprocess(batch):
    # Tokenize text
    encodings = tokenizer(
        batch["text"],
        max_length=256,
        truncation=True
    )
    # Explicitly ensure labels are passed as standard integers
    encodings["labels"] = [int(x) for x in batch["label_id"]]
    return encodings

# Remove ALL raw text/string columns so the collator only sees numeric tensors
columns_to_remove = split_datasets["train"].column_names

tokenized = split_datasets.map(
    preprocess,
    batched=True,
    remove_columns=columns_to_remove  # Strips 'text', 'label', and 'label_id'
)

# -------------------------------------------------------------
# 2. Custom Classification Wrapper with Aligned Weights
# -------------------------------------------------------------
class LFM25ForSequenceClassification(nn.Module):
    def __init__(self, model_id, num_labels, config):
        super().__init__()
        self.num_labels = num_labels
        self.config = config

        # AutoModelForMaskedLM correctly matches the 'lfm2.*' keys in the checkpoint
        mlm_model = AutoModelForMaskedLM.from_pretrained(
            model_id,
            config=config,
            trust_remote_code=True
        )
        # Extract the underlying bidirectional encoder body (no UNEXPECTED/MISSING warnings)
        self.encoder = mlm_model.lfm2
        del mlm_model.lm_head  # Free MLM projection memory

        hidden_size = config.hidden_size  # 1024

        # Classification projection head
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)

        # Extract representations [batch_size, seq_len, hidden_size]
        last_hidden_state = outputs[0]

        # Mean Pooling over unmasked tokens
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states if hasattr(outputs, "hidden_states") else None,
            attentions=outputs.attentions if hasattr(outputs, "attentions") else None,
        )

# -------------------------------------------------------------
# 3. Model Initialization
# -------------------------------------------------------------
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.num_labels = len(ACTION_LABELS)
config.label2id = label2id
config.id2label = id2label

model = LFM25ForSequenceClassification(
    model_id=MODEL_ID,
    num_labels=len(ACTION_LABELS),
    config=config
)

# -------------------------------------------------------------
# 4. Metrics & Evaluation
# -------------------------------------------------------------
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=preds, references=labels)

# -------------------------------------------------------------
# 5. Trainer Configuration (Using tokenized["test"])
# -------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="/content/runs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],  # Fixed: Points to 'test' split produced by train_test_split
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8),
    compute_metrics=compute_metrics
)

# -------------------------------------------------------------
# 6. Train & Export Checkpoint
# -------------------------------------------------------------
trainer.train()



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 52.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 19.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.11 GiB is allocated by PyTorch, and 301.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Save final checkpoint to Google Drive
torch.save(model.state_dict(), f"{OUTPUT_DIR}/pytorch_model.bin")
config.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"[SUCCESS] Pre-trained checkpoint permanently saved to: {OUTPUT_DIR}")

[SUCCESS] Pre-trained checkpoint permanently saved to: /content/drive/MyDrive/lfm25_encoder_checkpoint
